# --- Install Dependencies ---

In [113]:
!pip install langchain langchain_community langchain_openai langgraph \
           faiss-cpu sentence_transformers pypdf openpyxl streamlit --quiet


print('✅ All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 107.3 MB/s eta 0:00:00
✅ All packages installed.


# --- Set the OpenAI key ---

In [2]:
import os


# Option A — Colab Secrets
try:
   from google.colab import userdata
   os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
   print('✅ OpenAI API key loaded from Colab Secrets.')
except Exception:
   # Option B — paste directly
   os.environ['OPENAI_API_KEY'] = 'sk-...'   # ← Replace with your key
   print('⚠️  Key set directly — do not share this notebook.')

✅ OpenAI API key loaded from Colab Secrets.


# --- Load Patient Registry / Lookup Index from records.xlsx ---

In [51]:
import openpyxl
import pandas as pd


wb = openpyxl.load_workbook('records.xlsx')
ws = wb.active

# Extract headers from excel worksheet
headers = [cell.value for cell in ws[1]]
# Convert rows into structured dicts for downstream agent use
rows = [dict(zip(headers, row)) for row in ws.iter_rows(min_row=2, values_only=True)]


# Build lookup dict by name and phone
PATIENT_REGISTRY = {}
for r in rows:
   if r.get('Name'):
       PATIENT_REGISTRY[r['Name'].lower().strip()] = r
       if r.get('Phone_number'):
           PATIENT_REGISTRY[str(r['Phone_number']).strip()] = r


df_patients = pd.DataFrame(rows)
print(f'✅ Loaded {len(rows)} patient records\n')
print(df_patients[['Name','Age','Gender','Address','Summary']].to_string(index=False))

✅ Loaded 7 patient records

           Name  Age Gender                            Address                                                                                                                                                                                                                                                      Summary
     Rahul Negi   31   Male                         Chattarpur                                                                                                                                                                                              Rahul is a fit and healthy person. He is doing well in his life
   Rebeca Nagle   36 Female   9125 XYZ Hill St, Tigard,OR97223                                                                                                                                                                                                                                                         None
   Rebeca Nagle   36 Fem

In [4]:
type(df_patients)

pandas.core.frame.DataFrame

In [6]:
df_patients

,Phone_number,Email,Name,Age,Gender,Address,Summary
0,7982179305,rahul16negi@gmail.com,Rahul Negi,31,Male,Chattarpur,Rahul is a fit and healthy person. He is doing...
1,+1-541-950-0000,None,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",None
2,+1-541-950-0000,None,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",None
3,+1-541-950-0000,None,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",None
4,+91-98220-45322,None,Ramesh Kulkarni,65,Male,"52 Residency Road, Chennai",Patient presents for routine checkup with a hi...
5,+91-98180-11245,None,Anjali Mehra,33,Female,"202 Lakeview Apartments, Pune",Patient presents with 5-day history of dry cou...
6,+91-98450-11223,None,David Thompson,51,Male,"17 MG Road, Indiranagar, Bangalore",Patient presents for follow-up of Type 2 Diabe...


# --- Load Patient Data into FAISS Vector Store ---
RAG Pipeline
1. Parse 4 patient PDFs with pypdf
2. Split into overlapping text chunks
3. Embed with OpenAIEmbeddings  - OPTIONAL: Use sentence_transformers as local, no API cost option
4. Store in FAISS for semantic search

When the agent needs a patient's history it queries this vector store.

In [52]:
# Basic RAG Setup
from pypdf import PdfReader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

## Atlernative embedding models:
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS



In [53]:
# Loading the PDFs

PDF_FILES = {
    "Rebeca Nagle": "sample_patient.pdf",
    "Anjali Mehra": "sample_report_anjali.pdf",
    "David Thompson": "sample_report_david.pdf",
    "Ramesh Kulkarni": "sample_report_ramesh.pdf"
}

docs = []

# Loop through the files and load them
for patient_name, file_path in PDF_FILES.items():
    loader = PyPDFLoader(file_path)
    # Load pages and add patient name to metadata so we know who is who
    patient_docs = loader.load()
    for d in patient_docs:
        d.metadata["patient_name"] = patient_name
    docs.extend(patient_docs)

print("Total Pages Loaded:", len(docs))
if len(docs) > 0:
    print("Sample Text:", docs[0].page_content[:50])
    print("Sample Metadata:", docs[0].metadata)

Total Pages Loaded: 15
Sample Text: Rebeca Nagle 
9/7/2022 8:00 AM 
Location: Bridport
Sample Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-01-22T09:53:45-05:00', 'title': 'History and Physical Note', 'moddate': '2025-01-22T09:53:45-05:00', 'source': 'sample_patient.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'patient_name': 'Rebeca Nagle'}


In [54]:
# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=160,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print(f"Created {len(all_splits)} chunks from {len(docs)} pages.")

Created 32 chunks from 15 pages.


In [ ]:
# Embedding + Vector Store
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

api_key = userdata.get('OPENAI_API_KEY')

# Option 1: OpenAI (Recommended for this setup)
embedding = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=api_key)

# Option 2: Local HuggingFace (Uncomment to use instead of OpenAI)
# embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create Vector store and pass the chunked/split data and the wrapped embedding model
vectorstore_FAISS = FAISS.from_documents(all_splits, embedding)

print("Vector store created successfully with FAISS.")

✅ Vector store created successfully with FAISS.


In [ ]:
# Creating Retriever ("R" of RAG)
retriever = vectorstore_FAISS.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4} # Retrieve top 4 most relevant chunks
)

print("Retriever initialized.")

✅ Retriever initialized.


In [ ]:
# Model & parser setup.
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o", temperature=0)

parser = StrOutputParser()

test = llm.invoke('Say LLM ready and nothing else.')
print(f'LLM connected: {test.content}')

# print("LLM model initialized and parser selected. Ready for chaining.")

✅ LLM connected: LLM ready


# --- Creating Tools ---

In [65]:
import random
from datetime import datetime, timedelta

## Tool 1 - Patient Lookup

In [66]:
def lookup_patient(key: str):
    key = key.lower().strip()
    matches = []

    for k, v in PATIENT_REGISTRY.items():
        score = 0

        if key == k:
            score = 100
        elif key in k:
            score = 80
        elif k in key:
            score = 70

        if score > 0:
            if k.startswith(key):
                score += 10

            length_gap = abs(len(k) - len(key))
            score -= min(length_gap, 20)

            matches.append((score, v))

    matches.sort(key=lambda x: x[0], reverse=True)

    results = []
    seen_names = set()

    for score, patient in matches:
        name = patient["Name"]
        if name not in seen_names:
            seen_names.add(name)
            results.append({
                #"match_score": score,
                "name": patient["Name"],
                "age": patient["Age"],
                "gender": patient["Gender"],
                "phone": str(patient["Phone_number"]),
                "address": patient["Address"],
                "summary": patient["Summary"],
            })

    if results:
        return {
            "status": "found",
            **results[0],
            "note": "Best match selected based on fuzzy search score"
        }

    return {"status": "not_found", "matches": []}


## Tool 2 - retrieve_medical_history

In [67]:
# ── TOOL 2: retrieve_medical_history ──────────────────────────────────────
# Semantic search over patient PDFs + LLM summarization
def retrieve_medical_history(patient_name: str) -> dict:
    query = f"medical history diagnosis medications treatment plan for {patient_name}"
    docs = retriever.invoke(query)

    normalized_name = patient_name.lower().strip()
    relevant = [
        d for d in docs
        if d.metadata.get("patient_name", "").lower().strip() == normalized_name
    ]

    if not relevant:
        return {
            "status": "not_found",
            "patient": patient_name,
            "summary": "No confidently matched clinical notes were found for this patient."
        }

    context = "\n\n".join(d.page_content for d in relevant)

    prompt = ChatPromptTemplate.from_template(
        "You are a medical summarization assistant.\n"
        "Summarize the clinical notes for patient: {patient_name}\n"
        "Include: Diagnoses, Medications, Lab Results (if any), Vitals, Treatment Plan.\n\n"
        "Clinical Notes:\n{context}\n\nProvide a clear, structured clinical summary."
    )

    summary = (prompt | llm | parser).invoke({
        "patient_name": patient_name,
        "context": context
    })

    return {"status": "found", "patient": patient_name, "summary": summary}

## Tool 3 - book_appointment

In [68]:
import random
from datetime import datetime, timedelta

# --- DOCTOR DATABASE ---
DOCTOR_DB = {
   'general': [{'name':'Dr. Priya Sharma', 'hospital':'City Health Clinic'}],
   'nephrology': [{'name':'Dr. Anand Mehta', 'hospital':'Apollo Hospital'}],
   'endocrinology': [{'name':'Dr. Sunita Rao', 'hospital':'Fortis Hospital'}],
   'cardiology': [{'name':'Dr. Vikram Nair', 'hospital':'Max Hospital'}],
   'pulmonology': [{'name':'Dr. Kavita Joshi', 'hospital':'Columbia Asia'}],
   'family medicine': [{'name':'Dr. Megana Lanoi', 'hospital':'Bridport Family Medicine'}],
}

SPEC_MAP = {
   'nephrologist':'nephrology', 'kidney':'nephrology', 'renal':'nephrology',
   'diabetologist':'endocrinology', 'diabetes':'endocrinology', 'endocrinologist':'endocrinology',
   'cardiologist':'cardiology', 'heart':'cardiology',
   'pulmonologist':'pulmonology', 'lung':'pulmonology',
   'family':'family medicine', 'primary care':'family medicine'
}

def book_appointment(patient_name: str, specialty: str, preference: str='morning') -> dict:
   """Core logic for scheduling an appointment."""
   print(f"[Internal Tool Log] Attempting booking: Patient={patient_name}, Specialty={specialty}")
   spec_key = specialty.lower().strip()
   resolved_spec = spec_key

   for keyword, mapping in SPEC_MAP.items():
       if keyword in spec_key:
           resolved_spec = mapping
           break

   match = DOCTOR_DB.get(resolved_spec)
   if not match:
       for key in DOCTOR_DB.keys():
           if key in spec_key or spec_key in key:
               match = DOCTOR_DB[key]
               resolved_spec = key
               break

   if not match:
       match = DOCTOR_DB['general']
       resolved_spec = 'general'

   doc = random.choice(match)
   date_obj = datetime.now() + timedelta(days=random.randint(2, 5))
   date_str = date_obj.strftime('%A, %d %B %Y')
   time_str = '10:30 AM' if 'morning' in preference.lower() else '3:00 PM'
   ref = f'APT-{random.randint(10000, 99999)}'

   res = {
       'status': 'confirmed',
       'booking_ref': ref,
       'patient': patient_name,
       'doctor': doc['name'],
       'hospital': doc['hospital'],
       'specialty': resolved_spec,
       'date': date_str,
       'time': time_str
   }
   print(f"[Internal Tool Log] Successfully generated ref: {ref}")
   return res

## Tool 4 - search_medical_info

In [69]:
# ── TOOL 4: search_medical_info ───────────────────────────────────────────
# FAISS retrieval + LLM summarization of medical guidelines
def search_medical_info(topic: str) -> dict:
   docs    = retriever.invoke(f'treatment guidelines diagnosis {topic}')
   context = '\n\n'.join(d.page_content for d in docs)
   prompt  = ChatPromptTemplate.from_template(
       'You are a clinical information assistant.\n'
       'Provide a concise summary for: {topic}\n'
       'Include: Overview, Key Treatments, Medications, Red Flags.\n\n'
       'Available context:\n{context}\n\n'
       'Use context where relevant; supplement with clinical knowledge. '
       'Respond in clear bullet points.'
   )
   result = (prompt | llm | parser).invoke({'topic':topic,'context':context})
   return {'status':'found','topic':topic,'information':result}


## Tool 5 - update_patient_summary

In [70]:
# ── TOOL 5: update_patient_summary ────────────────────────────────────────
# Writes back to in-memory registry | Production: PATCH /fhir/Patient/<id>
def update_patient_summary(patient_name: str, new_summary: str) -> dict:
   key = patient_name.lower().strip()
   if key in PATIENT_REGISTRY:
       PATIENT_REGISTRY[key]['Summary'] = new_summary
       return {'status':'updated','patient':patient_name}
   return {'status':'not_found','message':f'Patient {patient_name} not found.'}

In [ ]:
print('All 5 tools defined.')
print('   Smoke test — lookup_patient(Anjali Mehra):')
print(lookup_patient('Anjali Mehra'))

✅ All 5 tools defined.
   Smoke test — lookup_patient(Anjali Mehra):
{'status': 'found', 'name': 'Anjali Mehra', 'age': 33, 'gender': 'Female', 'phone': '+91-98180-11245', 'address': '202 Lakeview Apartments, Pune', 'summary': 'Patient presents with 5-day history of dry cough and mild fever, diagnosed with Upper Respiratory Infection (J06.9), and advised symptomatic management with antihistamines and fluids, rest, and follow-up in 5 days.', 'note': 'Best match selected based on fuzzy search score'}


# --- LangGraph Agent (StateGraph) ---
### The agent uses 'LangGraph' - the framework in 'requirements.txt'.

In [ ]:
import operator
import json
import logging
from typing import Annotated, TypedDict, List, Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool


logger = logging.getLogger(__name__)


# ── 1. System Prompt ─────────────────────────────────────────────────────────
# Central place for all orchestration guidance — keeps tool docstrings clean.

SYSTEM_PROMPT = """You are a clinical AI assistant for a healthcare platform. \
You help staff retrieve patient records, review medical histories, schedule \
appointments, search treatment guidelines, and update patient summaries.

Guidelines:
- Call lookup_patient_tool first when given a partial or unverified name.
- Use the full name returned by lookup_patient_tool for all subsequent patient tools.
- Never fabricate clinical data. If a tool returns no result, say so clearly.
- Respond in a clear, professional tone appropriate for clinical staff.
- When booking appointments, confirm specialty and timing if the query is ambiguous."""


# ── 2. State ──────────────────────────────────────────────────────────────────

class AgentState(TypedDict):
    messages:       Annotated[List[BaseMessage], operator.add]
    query:          str
    patient_name:   Optional[str]
    tool_results:   Annotated[list, operator.add]
    final_response: Optional[str]


# ── 3. Tools ──────────────────────────────────────────────────────────────────

@tool
def lookup_patient_tool(name: str) -> dict:
    """Look up basic patient demographic information by name."""
    return lookup_patient(name)

@tool
def medical_history_tool(patient_name: str) -> dict:
    """Retrieve clinical summaries and history from patient records."""
    return retrieve_medical_history(patient_name)

@tool
def book_appointment_tool(patient_name: str, specialty: str, preference: str = "morning") -> dict:
    """Schedule a new appointment for a patient with a given specialty and time preference."""
    return book_appointment(patient_name, specialty, preference)

@tool
def medical_info_search_tool(topic: str) -> dict:
    """Search medical literature for general guidelines and treatment information."""
    return search_medical_info(topic)

@tool
def update_summary_tool(patient_name: str, new_summary: str) -> dict:
    """Update the clinical summary for a patient in the registry."""
    return update_patient_summary(patient_name, new_summary)


TOOLS = [
    lookup_patient_tool,
    medical_history_tool,
    book_appointment_tool,
    medical_info_search_tool,
    update_summary_tool,
]

tool_node = ToolNode(TOOLS)


# ── 4. Model ──────────────────────────────────────────────────────────────────
# temperature=0: deterministic outputs are important in a clinical context.

model = ChatOpenAI(model="gpt-4o", temperature=0).bind_tools(TOOLS)


# ── 5. Node: agent ───────────────────────────────────────────────────────────


def agent_node(state: AgentState) -> dict:
    """Invoke the LLM. Prepend system prompt on first turn; emit tool calls or finish."""
    messages = state["messages"]

    # Injects SYSTEM_PROMPT on the first turn only (not every loop pass).
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = model.invoke(messages)
    return {"messages": [response]}


# ── 6. Node: synthesize ───────────────────────────────────────────────────────
# Utilizes a dedicated formatting pass before the graph exits. Produces the clean,
# user-facing string that run_agent reads from final_response.

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a clinical documentation assistant. Compose a single, clear, "
     "professional response for clinical staff from the conversation below.\n\n"
     "Rules:\n"
     "- Synthesize all tool results into one coherent reply\n"
     "- Present medical history in readable clinical format\n"
     "- Confirm appointment details explicitly if one was booked\n"
     "- Present treatment guidelines as labelled bullet points\n"
     "- If a tool returned an error or no data, state that plainly"),
    ("human", "Original query: {query}\n\nConversation:\n{messages}"),
])

def synthesize_node(state: AgentState) -> dict:
    """Format the accumulated conversation into a clean final clinical response."""
    message_log = "\n".join(
        f"[{type(m).__name__}] {getattr(m, 'content', '')}"
        for m in state["messages"]
    )
    chain    = SYNTHESIS_PROMPT | ChatOpenAI(model="gpt-4o", temperature=0)
    response = chain.invoke({
        "query":    state.get("query", ""),
        "messages": message_log,
    })
    return {"final_response": response.content}


# ── 7. Routing ────────────────────────────────────────────────────────────────

def route_after_agent(state: AgentState) -> str:
    """Continue looping through tools, or hand off to synthesis."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "synthesize"


# ── 8. Capture Patient ────────────────────────────────────────────────────────────────


def capture_patient_node(state: AgentState) -> dict:
    """
    Inspect the latest tool message and update patient_name when a patient
    has been resolved by a tool.
    """
    if not state.get("messages"):
        return {}

    last_message = state["messages"][-1]
    raw_content = getattr(last_message, "content", "")

    # ToolMessage.content is often a string, even when the tool returned a dict.
    data = None
    if isinstance(raw_content, str):
        try:
            data = json.loads(raw_content)
        except Exception:
            return {}
    elif isinstance(raw_content, dict):
        data = raw_content

    if not isinstance(data, dict):
        return {}

    # lookup_patient_tool returns {"status": "found", "name": "...", ...}
    # retrieve_medical_history returns {"status": "found", "patient": "...", ...}
    candidate = data.get("name") or data.get("patient")

    if isinstance(candidate, str) and candidate.strip():
        return {
            "patient_name": candidate.strip(),
            "tool_results": [data],
        }

    # Still capture raw tool result if useful
    return {"tool_results": [data]}


# ── 9. Graph ──────────────────────────────────────────────────────────────────

agent_graph = StateGraph(AgentState)

agent_graph.add_node("agent",      agent_node)
agent_graph.add_node("tools",      tool_node)
agent_graph.add_node("capture_patient", capture_patient_node)
agent_graph.add_node("synthesize", synthesize_node)

agent_graph.set_entry_point("agent")

agent_graph.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "synthesize": "synthesize"},
)
agent_graph.add_edge("tools", "capture_patient")
agent_graph.add_edge("capture_patient", "agent")
agent_graph.add_edge("synthesize", END)


medical_agent = agent_graph.compile()

logger.info("Medical agent compiled — flow: agent ⇄ tools → synthesize → END")

# --- 8. Patient Memory Module (FAISS-backed) ---

In [92]:
from datetime import datetime
from langchain_community.vectorstores import FAISS


# ── Session Memory ────────────────────────────────────────────────────────────
# Keys are normalised to lowercase so save() and retrieve() always agree.
# Each patient gets their own isolated FAISS index.

class PatientMemory:
    def __init__(self):
        self._stores: dict = {}
        self._embeddings    = embedding  # embedding model defined earlier in notebook

    def save(self, patient_name: str, query: str, response: str) -> None:
        """Persist a query/response pair for a patient."""
        key  = patient_name.lower().strip()
        text = f"[Session] Q: {query} | A: {response[:400]}"
        meta = [{"patient": patient_name, "ts": datetime.now().isoformat()}]

        if key not in self._stores:
            self._stores[key] = FAISS.from_texts([text], self._embeddings, metadatas=meta)
        else:
            self._stores[key].add_texts([text], metadatas=meta)

        session_count = len(self._stores[key].index_to_docstore_id)
        logger.info('Memory saved for "%s" (%d session(s) stored)', key, session_count)

    def retrieve(self, patient_name: str, query: str, k: int = 3) -> str:
        """Return the k most relevant past sessions for a patient."""
        key = patient_name.lower().strip()
        if key not in self._stores:
            return f'No prior session history found for "{patient_name}".'
        docs = self._stores[key].similarity_search(query, k=k)
        return "\n\n".join(f"[{i + 1}] {d.page_content}" for i, d in enumerate(docs))

    def show_all(self, patient_name: str) -> None:
        """Print every stored session for a patient."""
        key = patient_name.lower().strip()
        if key not in self._stores:
            logger.warning('No memory found for "%s".', patient_name)
            return
        docs = self._stores[key].similarity_search(
            "session query medication diagnosis", k=20
        )
        logger.info('Memory for "%s" — %d session(s) found', patient_name, len(docs))
        print("=" * 65)
        for i, doc in enumerate(docs, 1):
            print(f"\n[Session {i}]")
            print(doc.page_content)
        print("=" * 65)


patient_memory = PatientMemory()
logger.info("PatientMemory ready — keys stored in lowercase for consistent lookup.")

# 9. Full Agent Runner
### Single entry point. Runs the LangGraph agent, injects memory, saves session.

In [ ]:
# ── Agent Runner ──────────────────────────────────────────────────────────────
# Invokes medical_agent, extracts the synthesized response from final_response,
# and persists the exchange to patient_memory when a patient was identified.

from langchain_core.messages import HumanMessage

def run_agent(query: str, verbose: bool = True) -> str:
    """Run the medical agent for a query and return the final clinical response."""
    logger.info("Query received: %s", query)

    if verbose:
        print("\n" + "=" * 65)
        print(f"QUERY: {query}")
        print("=" * 65)

    final_state = medical_agent.invoke(
        {
          "messages":       [HumanMessage(content=query)],
          "query":          query,
          "patient_name":   None,
          "current_task":   None,
          "tool_results":   [],
          "final_response": None,
        },
    # recursion_limit: caps agent↔tool iterations; raise if complex queries need more.
      config={"recursion_limit": 15}
    )

    ## Uncomment these for debugging
    #print("FINAL STATE PATIENT:", final_state.get("patient_name"))
    # print("\nDEBUG: MESSAGE TRACE")
    # print("-" * 65)
    # for i, m in enumerate(final_state["messages"]):
    #     print(f"\n[{i}] {type(m).__name__}")
    #     print(getattr(m, "content", ""))

    response    = final_state["final_response"] or ""
    patient     = final_state.get("patient_name") or ""
    patient_key = patient.lower().strip()

    # Persist to memory only when a real patient was resolved
    if patient_key and patient_key not in ("unknown", "", "none"):
        patient_memory.save(patient, query, response)

    if verbose:
        print("\n" + "-" * 65)
        print("FINAL RESPONSE:")
        print("-" * 65)
        print(response)
        print("=" * 65)

    return response


logger.info("run_agent() ready.")

# Live Demos Queries

In [104]:
r1 = run_agent('What is Anjali Mehra\'s diagnosis and treatment plan?')


QUERY: What is Anjali Mehra's diagnosis and treatment plan?
FINAL STATE PATIENT: Anjali Mehra

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
**Patient Information:**
- **Name:** Anjali Mehra
- **Age:** 33
- **Gender:** Female
- **Contact:** +91-98180-11245
- **Address:** 202 Lakeview Apartments, Pune

**Medical History:**
- **Presenting Symptoms:** 5-day history of dry cough and mild fever
- **Diagnosis:** Upper Respiratory Infection (J06.9)

**Treatment Plan:**
- **Symptomatic Management:**
  - **Antihistamines:** To alleviate symptoms
  - **Increased Fluid Intake:** To maintain hydration
  - **Rest:** To support recovery
- **Follow-Up:** Scheduled in 5 days

No appointment details were explicitly confirmed in the provided information.



=================================================================
QUERY: What is Anjali Mehra's diagnosis and treatment plan?
=================================================================
FINAL STATE PATIENT: Anjali Mehra

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
**Patient Information:**
- **Name:** Anjali Mehra
- **Age:** 33
- **Gender:** Female
- **Contact:** +91-98180-11245
- **Address:** 202 Lakeview Apartments, Pune

**Medical History:**
- **Presenting Symptoms:** 5-day history of dry cough and mild fever
- **Diagnosis:** Upper Respiratory Infection (J06.9)

**Treatment Plan:**
- **Symptomatic Management:**
  - **Antihistamines:** To alleviate symptoms
  - **Increased Fluid Intake:** To maintain hydration
  - **Rest:** To support recovery
- **Follow-Up:** Scheduled in 5 days

No appointment details were explicitly confirmed in the provided information.
=================================================================


In [96]:
r2 = run_agent('Book an endocrinologist appointment for David Thompson.')


QUERY: Book an endocrinologist appointment for David Thompson.
[Internal Tool Log] Attempting booking: Patient=David Thompson, Specialty=endocrinology
[Internal Tool Log] Successfully generated ref: APT-58302

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
**Patient Information:**

- **Name:** David Thompson
- **Age:** 51
- **Gender:** Male
- **Phone:** +91-98450-11223
- **Address:** 17 MG Road, Indiranagar, Bangalore

**Medical History:**

- **Condition:** Type 2 Diabetes Mellitus (E11.9)
- **Symptoms:** Increased thirst and urination
- **Current Plan:**
  - Increase metformin dosage
  - Order HbA1c and Lipid Profile
  - Schedule nutritionist follow-up
  - Return in 3 months

**Appointment Details:**

- **Doctor:** Dr. Sunita Rao
- **Specialty:** Endocrinology
- **Hospital:** Fortis Hospital
- **Date:** Sunday, 19 April 2026
- **Time:** 10:30 AM
- **Booking Reference:** APT-58302

The

In [106]:
r3 = run_agent(
   'Get Ramesh Kulkarni\'s medical history, book him a cardiology appointment, '
   'and give me the latest treatment guidelines for hypertension.'
)



QUERY: Get Ramesh Kulkarni's medical history, book him a cardiology appointment, and give me the latest treatment guidelines for hypertension.
[Internal Tool Log] Attempting booking: Patient=Ramesh Kulkarni, Specialty=cardiology
[Internal Tool Log] Successfully generated ref: APT-54062
FINAL STATE PATIENT: Ramesh Kulkarni

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
Here is the information you requested:

### Ramesh Kulkarni's Medical History:
- **Diagnoses:** Essential Hypertension (I10)
- **Medications:** Telmisartan 40mg once daily (OD)
- **Lab Results:** Routine labs have been ordered (results not available in current notes).
- **Vitals:** Blood Pressure: 130/84 mmHg, Pulse: 76 beats per minute, Temperature: 98.4°F
- **Treatment Plan:** Continue current medication (Telmisartan 40mg OD), recommend lifestyle modifications, and schedule a follow-up visit in 6 months.
- **Additional

In [109]:
r4 = run_agent(
   'My 70-year-old father has chronic kidney disease. '
   'I want to book a nephrologist for him. '
   'Also, can you summarize the latest treatment methods?'
)



QUERY: My 70-year-old father has chronic kidney disease. I want to book a nephrologist for him. Also, can you summarize the latest treatment methods?

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
Medical History:
- Patient Age: 70 years
- Condition: Chronic Kidney Disease (CKD)

Appointment Details:
- No appointment has been booked yet. Please provide the patient's name and preferred appointment date and time to proceed with scheduling a nephrologist consultation.

Latest Treatment Methods for Chronic Kidney Disease:
- **Blood Pressure Control**: Maintain blood pressure below 130/80 mmHg using ACE inhibitors or ARBs.
- **Blood Sugar Management**: For diabetic patients, aim for an HbA1c level of around 7%.
- **Dietary Modifications**: Limit protein, sodium, potassium, and phosphorus intake as per dietary guidelines.
- **Medication**: Use of medications like statins for cholesterol man

In [110]:
r5 = run_agent(
   'Summarize Rebeca Nagle\'s full medical history '
   'including her lab results, diagnoses, and visit summaries.'
)



QUERY: Summarize Rebeca Nagle's full medical history including her lab results, diagnoses, and visit summaries.

-----------------------------------------------------------------
FINAL RESPONSE:
-----------------------------------------------------------------
Here is the summarized medical history for Rebeca Nagle:

**Clinical Summary for Patient: Rebeca Nagle**

**Patient Information:**
- Name: Rebeca Nagle
- Date of Birth: May 19, 1986
- Gender: Female
- Contact: 9125 XYZ Hill St, Tigard, OR 97223
- Phone: +1-541-950-0000

**Diagnoses:**
- Allergies
- Joint pain

**Medications:**
- No specific medications listed in the notes.

**Lab Results:**
- Last lab work was conducted in 2020; screening labs have been requested during the latest visit.

**Vitals:**
- No specific vitals recorded in the notes.

**Past Medical History:**
- Major Events: C-section, Tonsillectomy
- Preventive Care: PAP smear negative as of September 2022
- Family Health History: Rheumatoid Arthritis (RA) in family


# UI Interface

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
from datetime import datetime

# -------------------------------------------------
# Page config
# -------------------------------------------------
st.set_page_config(
    page_title="Healthcare AI Assistant",
    page_icon="🩺",
    layout="wide",
    initial_sidebar_state="expanded",
)

# -------------------------------------------------
# Theme / light custom styling
# -------------------------------------------------
st.markdown(
    """
    <style>
        .main {
            background: linear-gradient(180deg, #f8fbff 0%, #f3f7fc 100%);
        }
        .block-container {
            padding-top: 1.5rem;
            padding-bottom: 2rem;
            max-width: 1300px;
        }
        .hero-card {
            background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%);
            color: white;
            padding: 1.4rem 1.5rem;
            border-radius: 22px;
            box-shadow: 0 14px 35px rgba(15, 23, 42, 0.18);
            margin-bottom: 1rem;
        }
        .metric-card {
            background: white;
            border: 1px solid rgba(148, 163, 184, 0.18);
            border-radius: 20px;
            padding: 1rem 1rem 0.8rem 1rem;
            box-shadow: 0 8px 22px rgba(15, 23, 42, 0.06);
        }
        .section-card {
            background: rgba(255,255,255,0.88);
            border: 1px solid rgba(148, 163, 184, 0.15);
            border-radius: 22px;
            padding: 1.15rem;
            box-shadow: 0 10px 24px rgba(15, 23, 42, 0.06);
        }
        .label-soft {
            display: inline-block;
            background: rgba(37, 99, 235, 0.10);
            color: #1d4ed8;
            font-weight: 600;
            padding: 0.28rem 0.65rem;
            border-radius: 999px;
            font-size: 0.8rem;
            margin-bottom: 0.6rem;
        }
        .tiny-muted {
            color: #64748b;
            font-size: 0.9rem;
        }
        .stTextInput > div > div,
        .stTextArea textarea,
        .stSelectbox > div > div,
        .stMultiSelect > div > div {
            border-radius: 14px !important;
        }
        .chat-bubble-user {
            background: #eff6ff;
            border: 1px solid #bfdbfe;
            padding: 0.9rem 1rem;
            border-radius: 16px;
            margin-bottom: 0.75rem;
        }
        .chat-bubble-assistant {
            background: white;
            border: 1px solid rgba(148, 163, 184, 0.18);
            padding: 0.9rem 1rem;
            border-radius: 16px;
            margin-bottom: 0.9rem;
            box-shadow: 0 6px 18px rgba(15, 23, 42, 0.05);
        }
        .tool-pill {
            display: inline-block;
            background: rgba(16, 185, 129, 0.12);
            color: #047857;
            padding: 0.2rem 0.6rem;
            border-radius: 999px;
            font-size: 0.76rem;
            font-weight: 600;
            margin-right: 0.35rem;
            margin-bottom: 0.35rem;
        }
        .footer-note {
            color: #64748b;
            text-align: center;
            padding-top: 0.5rem;
            font-size: 0.85rem;
        }
    </style>
    """,
    unsafe_allow_html=True,
)

# -------------------------------------------------
# Session state
# -------------------------------------------------
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "last_patient" not in st.session_state:
    st.session_state.last_patient = "None"

if "last_action" not in st.session_state:
    st.session_state.last_action = "Ready"

# -------------------------------------------------
# Replace this with your real backend function
# Example: from your_notebook_export import run_agent
# -------------------------------------------------
def run_agent_ui(query: str) -> str:
    """Connect Streamlit UI to the real LangGraph backend."""
    return run_agent(query)


# -------------------------------------------------
# Sidebar
# -------------------------------------------------
with st.sidebar:
    st.markdown("## 🩺 Healthcare AI")
    st.caption("Clinical assistant workspace")

    st.markdown("---")
    st.markdown("### Workspace")
    view = st.radio(
        "Choose view",
        ["Assistant", "Patient Snapshot", "System Status"],
        label_visibility="collapsed",
    )

    st.markdown("### Quick Tools")
    st.markdown('<span class="tool-pill">Patient Lookup</span>', unsafe_allow_html=True)
    st.markdown('<span class="tool-pill">Medical History</span>', unsafe_allow_html=True)
    st.markdown('<span class="tool-pill">Appointments</span>', unsafe_allow_html=True)
    st.markdown('<span class="tool-pill">Guidelines</span>', unsafe_allow_html=True)
    st.markdown('<span class="tool-pill">Summary Update</span>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("### Session")
    st.write(f"**Last patient:** {st.session_state.last_patient}")
    st.write(f"**Last action:** {st.session_state.last_action}")
    st.write(f"**Time:** {datetime.now().strftime('%b %d, %Y %I:%M %p')}")

    if st.button("Clear conversation", use_container_width=True):
        st.session_state.chat_history = []
        st.session_state.last_patient = "None"
        st.session_state.last_action = "Ready"
        st.rerun()

# -------------------------------------------------
# Header
# -------------------------------------------------
st.markdown(
    """
    <div class="hero-card">
        <div class="label-soft" style="background: rgba(255,255,255,0.15); color: #e0ecff;">AI-Enabled Clinical Workflow</div>
        <h1 style="margin:0; font-size: 2rem;">Modern Healthcare Assistant</h1>
        <p style="margin: 0.55rem 0 0 0; color: #dbeafe; font-size: 1rem;">
            Retrieve patient information, review medical history, book appointments, and surface treatment guidance in one clean workspace.
        </p>
    </div>
    """,
    unsafe_allow_html=True,
)

# -------------------------------------------------
# KPI Row
# -------------------------------------------------
col1, col2, col3, col4 = st.columns(4)
with col1:
    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
    st.metric("Agent Status", "Online")
    st.markdown('</div>', unsafe_allow_html=True)
with col2:
    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
    st.metric("Patients Resolved", len({m.get("patient") for m in st.session_state.chat_history if m.get("patient") and m.get("patient") != "None"}))
    st.markdown('</div>', unsafe_allow_html=True)
with col3:
    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
    st.metric("Conversation Turns", len(st.session_state.chat_history))
    st.markdown('</div>', unsafe_allow_html=True)
with col4:
    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
    st.metric("Backend", "LangGraph")
    st.markdown('</div>', unsafe_allow_html=True)

st.write("")

# -------------------------------------------------
# Main layouts
# -------------------------------------------------
if view == "Assistant":
    left, right = st.columns([1.65, 1])

    with left:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">Assistant Workspace</div>', unsafe_allow_html=True)
        st.subheader("Clinical Copilot")
        st.markdown('<p class="tiny-muted">Ask for patient records, history review, treatment guidance, or appointment scheduling.</p>', unsafe_allow_html=True)

        prompt = st.text_area(
            "Enter request",
            placeholder="Example: Show me the medical history for Anjali Mehra and summarize current treatment plan.",
            height=120,
        )

        c1, c2, c3 = st.columns([1, 1, 1.2])
        with c1:
            quick_patient = st.selectbox("Patient", ["None", "Anjali Mehra", "David Thompson", "Maria Garcia"])
        with c2:
            quick_action = st.selectbox("Action", ["Auto", "Lookup", "History", "Appointment", "Guidelines"])
        with c3:
            submit = st.button("Run Assistant", use_container_width=True, type="primary")

        if submit and (prompt or quick_patient != "None"):
            final_prompt = prompt.strip()
            if not final_prompt:
                final_prompt = f"Help with {quick_action.lower()} for {quick_patient}."
            elif quick_patient != "None" and quick_patient.lower() not in final_prompt.lower():
                final_prompt += f" Patient: {quick_patient}."

            result = run_agent_ui(final_prompt)

            patient_guess = quick_patient if quick_patient != "None" else st.session_state.last_patient
            st.session_state.chat_history.append(
                {
                    "user": final_prompt,
                    "assistant": result,
                    "patient": patient_guess,
                    "timestamp": datetime.now().strftime("%I:%M %p"),
                }
            )
            st.rerun()

        st.write("")
        st.markdown("### Conversation")

        if not st.session_state.chat_history:
            st.info("No interactions yet. Start with a patient lookup or medical history request.")
        else:
            for item in reversed(st.session_state.chat_history):
                st.markdown(
                    f'<div class="chat-bubble-user"><strong>You</strong><br>{item["user"]}<div class="tiny-muted" style="margin-top:0.35rem;">{item["timestamp"]}</div></div>',
                    unsafe_allow_html=True,
                )
                st.markdown(
                    f'<div class="chat-bubble-assistant"><strong>Assistant</strong><br>{item["assistant"]}</div>',
                    unsafe_allow_html=True,
                )

        st.markdown('</div>', unsafe_allow_html=True)

    with right:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">Quick Actions</div>', unsafe_allow_html=True)
        st.subheader("Suggested Requests")

        quick_prompts = [
            "Look up Anjali Mehra",
            "Show me the medical history for David Thompson",
            "Book a cardiology appointment for Maria Garcia tomorrow morning",
            "Search treatment guidelines for upper respiratory infection",
            "Update patient summary after reviewing today's notes",
        ]

        for qp in quick_prompts:
            if st.button(qp, use_container_width=True):
                result = run_agent_ui(qp)
                st.session_state.chat_history.append(
                    {
                        "user": qp,
                        "assistant": result,
                        "patient": st.session_state.last_patient,
                        "timestamp": datetime.now().strftime("%I:%M %p"),
                    }
                )
                st.rerun()

        st.write("")
        st.markdown("### Assistant Notes")
        st.markdown(
            """
            - Best for staff-facing workflows
            - Keeps replies structured and clinically readable
            - Designed to sit on top of your LangGraph toolchain
            """
        )
        st.markdown('</div>', unsafe_allow_html=True)

elif view == "Patient Snapshot":
    left, right = st.columns([1.2, 1])
    with left:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">Patient Overview</div>', unsafe_allow_html=True)
        st.subheader(st.session_state.last_patient if st.session_state.last_patient != "None" else "No Patient Selected")
        st.markdown("#### Demographics")
        if st.session_state.last_patient == "None":
            st.info("Resolve a patient through the assistant to populate this view.")
        else:
            st.write("**Patient resolved in session**")
            st.write(f"**Name:** {st.session_state.last_patient}")
            st.write("**Source:** LangGraph agent state / latest session context")

        st.markdown("#### Clinical Summary")
        st.info(
            "This panel is designed to show the currently resolved patient in session. You can extend it to call a dedicated patient snapshot tool or render structured tool output directly."
        )
        st.markdown('</div>', unsafe_allow_html=True)

    with right:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">Recent Activity</div>', unsafe_allow_html=True)
        st.subheader("Timeline")
        st.write("• Patient record resolved")
        st.write("• Clinical summary reviewed")
        st.write("• Follow-up recommended in 5 days")
        st.write("• Ready for appointment scheduling or summary update")
        st.markdown('</div>', unsafe_allow_html=True)

else:
    c1, c2 = st.columns([1, 1])
    with c1:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">System Status</div>', unsafe_allow_html=True)
        st.subheader("Environment")
        st.success("LangGraph agent available")
        st.success("Tooling layer configured")
        st.success("Streamlit UI loaded")
        st.markdown('</div>', unsafe_allow_html=True)

    with c2:
        st.markdown('<div class="section-card">', unsafe_allow_html=True)
        st.markdown('<div class="label-soft">Integration Notes</div>', unsafe_allow_html=True)
        st.subheader("Connect Your Backend")
        st.code(
            """# Replace placeholder with your real backend import
# Example:
# from healthcare_agent_backend import run_agent

result = run_agent(query)
""",
            language="python",
        )
        st.markdown('</div>', unsafe_allow_html=True)

st.markdown('<div class="footer-note">Designed for a modern clinical workflow: simple, clean, and easy to extend.</div>', unsafe_allow_html=True)
